<center>
<img src="../../img/ods_stickers.jpg">
## دورة التعلم الآلي المفتوحة
<center> المؤلف: [يوري كاشنيتسكي](https://www.linkedin.com/in/festline/)، عالم بيانات @ Mail.Ru Group <br>يتم توزيع جميع المحتوى بموجب ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/).



# <center> المهمة رقم 10 (تجريبي)
## <center> تعزيز التدرج
مهمتك هي التغلب على معيارين على الأقل في [مسابقة Kaggle Inclass](https://www.kaggle.com/c/flight-delays-spring-2018). هنا لن يتم تزويدك بتعليمات مفصلة. نقدم لك فقط وصفًا موجزًا ​​لكيفية تحقيق المعيار الثاني باستخدام Xgboost. نأمل، في هذه المرحلة من الدورة التدريبية، أن تقوم بإلقاء نظرة سريعة على البيانات لفهم أن هذا هو نوع المهمة التي سيؤدي فيها تعزيز التدرج بشكل جيد. على الأرجح سيكون Xgboost، ومع ذلك، لدينا الكثير من الميزات الفئوية هنا.
<img src='../../img/xgboost_meme.jpg' width=40% />


In [ ]:
import warnings

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

In [ ]:
train = pd.read_csv("../../data/flight_delays_train.csv")
test = pd.read_csv("../../data/flight_delays_test.csv")

In [ ]:
train.head()

In [ ]:
test.head()


بالنظر إلى وقت مغادرة الرحلة، ورمز شركة النقل، ومطار المغادرة، وموقع الوجهة، ومسافة الرحلة، يتعين عليك التنبؤ بتأخير المغادرة لأكثر من 15 دقيقة. كأبسط معيار، لنأخذ مصنف Xgboost وميزتين من الأسهل استخدامهما: DepTime وDistance. ينتج عن هذا النموذج 0.68202 على LB.


In [ ]:
X_train = train[["Distance", "DepTime"]].values
y_train = train["dep_delayed_15min"].map({"Y": 1, "N": 0}).values
X_test = test[["Distance", "DepTime"]].values

X_train_part, X_valid, y_train_part, y_valid = train_test_split(
    X_train, y_train, test_size=0.3, random_state=17
)


سنقوم بتدريب Xgboost باستخدام المعلمات الافتراضية على جزء من البيانات وتقدير ROC AUC.


In [ ]:
xgb_model = XGBClassifier(seed=17)

xgb_model.fit(X_train_part, y_train_part)
xgb_valid_pred = xgb_model.predict_proba(X_valid)[:, 1]

roc_auc_score(y_valid, xgb_valid_pred)


الآن نفعل الشيء نفسه مع مجموعة التدريب بأكملها، ونقوم بعمل تنبؤات لاختبار المجموعة وتشكيل ملف إرسال. هذه هي الطريقة التي تغلبت بها على المعيار الأول. 


In [ ]:
xgb_model.fit(X_train, y_train)
xgb_test_pred = xgb_model.predict_proba(X_test)[:, 1]

pd.Series(xgb_test_pred, name="dep_delayed_15min").to_csv(
    "xgb_2feat.csv", index_label="id", header=True
)


تم تحقيق المعيار الثاني في لوحة المتصدرين على النحو التالي:- تم أخذ الميزات `Distance` و`DepTime` دون تغيير
- تم إنشاء ميزة `Flight` من الميزات `Origin` و`Dest`
- الميزات `Month`، `DayofMonth`، `DayOfWeek`، `UniqueCarrier` و`Flight` تم تحويلها باستخدام OHE (`LabelBinarizer`)
- تم تدريب الانحدار اللوجستي وتعزيز التدرج (xgboost). تم ضبط معلمات Xgboost الفائقة عبر التحقق من الصحة. أولاً، تم تحسين المعلمات الفائقة المسؤولة عن تعقيد النموذج، ثم تم تثبيت عدد الأشجار عند 500 وتم ضبط خطوة التعلم.
- تم إجراء الاحتمالات المتوقعة عبر التحقق المتبادل باستخدام `cross_val_predict`. تم تعيين مزيج خطي من الانحدار اللوجستي وتنبؤات تعزيز التدرج في النموذج $w_1 * p_{logit} + (1 - w_1) * p_{xgb}$، حيث $p_{logit}$ هو احتمال من الفئة 1، تم التنبؤ به بواسطة الانحدار اللوجستي، و$p_{xgb}$ - نفس الشيء بالنسبة لـ xgboost. $w_1$ تم تحديد الوزن يدويًا.
- تم إجراء مجموعة مماثلة من التوقعات لمجموعة الاختبار. 
اتباع نفس الخطوات ليس إلزاميا. هذا مجرد وصف لكيفية تحقيق مؤلف هذه المهمة للنتيجة. ربما لا ترغب في اتباع نفس الخطوات، وبدلاً من ذلك، دعنا نقول، أضف بعض الميزات الجيدة وقم بتدريب غابة عشوائية مكونة من ألف شجرة.
حظ سعيد!